# Tagger Baseado em RNN

In [ ]:
# 1. Clona o projeto para o ambiente temporário do Colab
!git clone https://github.com/lucasaamorim/NLP

In [ ]:
# 2. Entra na pasta do projeto
%cd NLP

In [ ]:
# 3. Instala o arquivo de dependências (caso o notebook use algo além do TensorFlow padrão)
!uv pip install keras scikit-learn nltk regex

In [ ]:
import numpy as np
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from src.utils.data_loader import load_tagging_data
from src.utils.preprocessing import tokenize_sentences, vectorize_tags

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd /content/drive/MyDrive/pos_tagging_data/data

In [ ]:
# ln -s /caminho/real/no/drive /content/nome_que_o_codigo_espera
!ln -s "/content/drive/MyDrive/pos_tagging_data/data" "/content/NLP/data"

###Carregando os dados de POS Tagging

In [ ]:
data = load_tagging_data()
print("Dados do POS Tagging carregados")

###Separando os splits originais

In [ ]:
train_sentences = data["train"]["sentences"]
train_tags = data["train"]["tags"]

val_sentences = data["val"]["sentences"]
val_tags = data["val"]["tags"]

test_sentences = data["test"]["sentences"]
test_tags = data["test"]["tags"]

print(f"Sentenças de Treino: {len(train_sentences)} | Validação: {len(val_sentences)} | Teste: {len(test_sentences)}")

###Processando o Vocabulário de Palavras

In [ ]:
# vocab_size = 15000  # Limite máximo de palavras únicas
vectorizer, max_len = tokenize_sentences(train_sentences)

X_train = vectorizer([" ".join(s) for s in train_sentences]).numpy()
X_val = vectorizer([" ".join(s) for s in val_sentences]).numpy()
X_test = vectorizer([" ".join(s) for s in test_sentences]).numpy()

###Processando o Vocabulário de Tags

In [ ]:
tag_lookup, Y_train = vectorize_tags(train_tags, max_len)

num_tags = tag_lookup.vocabulary_size()

Y_val_list = []
for sentence_tags in val_tags:
    tag_ids = tag_lookup(sentence_tags).numpy()
    pad_length = max_len - len(tag_ids)
    if pad_length > 0:
        padded_tags = np.pad(tag_ids, (0, pad_length), constant_values=0) # 0 é o ID do <PAD>
    else:
        padded_tags = tag_ids[:max_len]
    Y_val_list.append(padded_tags)
Y_val = np.array(Y_val_list)

Y_test_list = []
for sentence_tags in test_tags:
    tag_ids = tag_lookup(sentence_tags).numpy()
    pad_length = max_len - len(tag_ids)
    if pad_length > 0:
        padded_tags = np.pad(tag_ids, (0, pad_length), constant_values=0)
    else:
        padded_tags = tag_ids[:max_len]
    Y_test_list.append(padded_tags)
Y_test = np.array(Y_test_list)

print(f"Formato final de X_train: {X_train.shape}")
print(f"Formato final de Y_train: {Y_train.shape}")
print(f"Formato final de X_test: {X_test.shape}")
print(f"Formato final de Y_test: {Y_test.shape}")
print(f"Número total de tags (classes): {num_tags}")

In [ ]:
# Hiperparâmetros
embedding_dim = 128
rnn_units = 128
BATCH_SIZE = 32
EPOCHS = 5

inputs = keras.Input(shape=(max_len,), dtype="int32")

rnn = layers.Embedding(
    input_dim=vectorizer.vocabulary_size(),
    output_dim=embedding_dim,
    mask_zero=True
)(inputs)

###Construindo o modelo RNN unidirecional

In [ ]:
rnn_unidirecional = layers.SimpleRNN(rnn_units, return_sequences=True)(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(rnn_unidirecional)

rnn_unidirecional_model = keras.Model(inputs, outputs)

rnn_unidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rnn_unidirecional_model.summary()

###Construindo o modelo LSTM unidirecional


In [ ]:
lstm_unidirecional = layers.LSTM(rnn_units, return_sequences=True)(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(lstm_unidirecional)

lstm_unidirecional_model = keras.Model(inputs, outputs)

lstm_unidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_unidirecional_model.summary()

###Construindo o modelo RNN bidirecional

In [ ]:
rnn_bidirecional = layers.Bidirectional(layers.SimpleRNN(rnn_units, return_sequences=True))(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(rnn_bidirecional)

rnn_bidirecional_model = keras.Model(inputs, outputs)

rnn_bidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rnn_bidirecional_model.summary()

###Construindo o modelo LSTM Bidirecional

In [ ]:
# RNN Bidirecional para capturar contexto tanto da esquerda-para-direita quanto da direita-para-esquerda
lstm_bidirecional = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True))(rnn)

# Camada densa distribuída no tempo (computa uma distribuição de probabilidade por palavra)
outputs = layers.Dense(num_tags, activation="softmax")(lstm_bidirecional)

lstm_bidirecional_model = keras.Model(inputs, outputs)

# Usamos SparseCategoricalCrossentropy porque nosso Y contém IDs inteiros (não one-hot vectors)
lstm_bidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_bidirecional_model.summary()

### Treinamento da RNN Unidirecional

In [ ]:
history = rnn_unidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da LSTM Unidirecional

In [ ]:
history = lstm_unidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da RNN Bidirecional

In [ ]:
history = rnn_bidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da LSTM Bidirecional

In [ ]:
history = lstm_bidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

###Função de Avaliação dos modelos

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def model_evaluation(model, X_test, Y_test, tag_lookup):
    # 1. Predict probabilities for each class per word
    predictions_prob = model.predict(X_test)

    # 2. Get the index of the class with the highest probability
    predicted_classes = np.argmax(predictions_prob, axis=-1)

    y_true_filtered = []
    y_pred_filtered = []

    # 3. Filter data by removing ID 0 (<PAD>)
    for i in range(len(Y_test)):
        for j in range(len(Y_test[i])):
            if Y_test[i][j] != 0:  # 0 is the <PAD> ID generated by StringLookup
                y_true_filtered.append(Y_test[i][j])
                y_pred_filtered.append(predicted_classes[i][j])

    # 4. Get the actual vocabulary from your tag_lookup
    full_vocabulary = tag_lookup.get_vocabulary()

    # Find which unique IDs actually remain in y_true to avoid scikit-learn errors
    present_ids = sorted(list(set(y_true_filtered)))

    # Map the corresponding names only to the IDs that appear in the test set
    filtered_names = [full_vocabulary[int(idx)] for idx in present_ids]

    print("\n" + "="*50)
    print("Classification Report Without Padding")
    print("="*50)
    print(classification_report(y_true_filtered, y_pred_filtered, labels=present_ids, target_names=filtered_names, zero_division=0))

    # 5. Confusion Matrix with correct names on the axes
    cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=present_ids)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=filtered_names, yticklabels=filtered_names)
    plt.title("Confusion Matrix (Without Padding)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

### Avaliação da RNN Unidirecional

In [ ]:
model_evaluation(rnn_unidirecional_model, X_test, Y_test, tag_lookup)

### Avaliação da LSTM Unidirecional

In [ ]:
model_evaluation(lstm_unidirecional_model, X_test, Y_test, tag_lookup)

### Avaliação da RNN Bidirecional

In [ ]:
model_evaluation(rnn_unidirecional_model, X_test, Y_test, tag_lookup)

### Avaliação da LSTM Bidirecional

In [ ]:
model_evaluation(lstm_bidirecional_model, X_test, Y_test, tag_lookup)